# Day 3: Clean Data 🧹

## 🎯 Today's Goal

Remove bad data and prepare the data for analysis!

**Time:** 2-3 hours  
**Difficulty:** ⭐⭐ Easy-Medium (using pandas filtering and grouping)

---

## 🔗 What You Learned Yesterday (Day 2)

**If you completed Day 2, you now know:**
- ✅ How to load TSV files (`pd.read_csv()`)
- ✅ What columns are in TCR-seq data (sequences, V/D/J genes, counts)
- ✅ How to explore DataFrames (`.head()`, `.shape`, `.describe()`)
- ✅ What each column means biologically

**Today, we'll take that SAME data file and clean it!**

---

## 📚 What You'll Learn Today

- Why we need to clean data (biology + CS reasons)
- How to filter out bad sequences (`df[df['column'] == 'value']`)
- How to aggregate duplicate sequences (`df.groupby().sum()`)
- How to prepare data for machine learning

**How This Builds On Day 2:**
- Yesterday: You loaded `patient_file.tsv` and explored it
- Today: You'll FILTER that same file (`df[df['frame'] == 'In']`)
- Tomorrow (Day 4): You'll use CLEANED files from multiple patients

**Data Flow:**
- **Day 2:** `patient_file.tsv` (raw) → Explored it
- **Day 3 (Today):** `patient_file.tsv` (raw) → Clean it → `patient_file_cleaned.tsv`
- **Day 4:** Multiple `*_cleaned.tsv` files → Compare them

---

## 🎓 Learning Progression (Like University Courses!)

**Think of this like learning Python:**
- **Day 2:** Learned to read files (`pd.read_csv()`)
- **Day 3 (Today):** Learning to filter data (`df[condition]`)
- **Day 4:** Learning to group/aggregate (`df.groupby()`)

**Think of this like learning biology:**
- **Day 2:** Looked at raw TCR sequences (some are broken)
- **Day 3 (Today):** Removing broken sequences (keeping only functional ones)
- **Day 4:** Comparing cleaned repertoires across patients

---

## 🧬 Biology Context: Why Clean Data?

### Why Filter to Productive Sequences?

**Biology reason:**
- Only "In" frame sequences actually function
- "Out" frame sequences are broken and can't recognize antigens
- **Think:** Like a broken key that doesn't fit any lock
- **Why it matters:** We only want to analyze functional T cells!

### Why Remove Invalid Amino Acids?

**Biology reason:**
- There are only 20 standard amino acids: A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y
- If a sequence contains other characters (like X, *, -, etc.), it's an error
- **Why it matters:** Invalid sequences are sequencing errors, not real TCRs

### Why Filter by Length?

**Biology reason:**
- Functional TCR sequences are usually 10-25 amino acids long
- Very short sequences (< 10) are incomplete or errors
- Very long sequences (> 25) are unusual and might be errors
- **Why it matters:** Unusual lengths are likely artifacts, not real TCRs

### Why Aggregate Duplicates?

**Biology reason:**
- The same amino acid sequence can come from different DNA sequences
- Multiple DNA sequences → same amino acid sequence = same TCR function
- We want to count how many T cells have this TCR, not how many DNA sequences
- **Why it matters:** We care about TCR function (amino acids), not DNA sequence

---

## 💻 Computer Science Context: Data Cleaning

### Why Clean Data?

**CS reasons:**
- **Remove errors:** Bad data causes problems in analysis
- **Standardize format:** Make data consistent
- **Reduce noise:** Focus on signal, not errors
- **Prepare for ML:** Machine learning needs clean data

### Common Cleaning Steps:

1. **Filtering:** Remove rows that don't meet criteria
   - Like: "Only keep rows where status = 'In'"
   - **CS:** `df[df['col'] == 'value']`

2. **Validation:** Check if data is valid
   - Like: "Does this string only contain valid characters?"
   - **CS:** Use `.apply()` with a validation function

3. **Aggregation:** Combine duplicate rows
   - Like: "Group by sequence, sum the counts"
   - **CS:** `df.groupby().agg()`

**In this project:** We do all three!

---

## 📄 Paper Reference

**Paper Section:** Methods - Data Preprocessing

**What the paper says:**
- "We filtered to productive (in-frame) TCR sequences"
- "Sequences with invalid amino acids were removed"
- "Sequences were filtered to length 10-25 amino acids"
- "Duplicate sequences were aggregated by amino acid sequence, V gene, and J gene"

**What this means:**
- We're doing exactly what the paper did!
- This is standard preprocessing for TCR-seq data
- Clean data = better analysis results

**Reference:** See `docs/paper.pdf` for full details

---

## 🤔 Questions to Think About (Before Starting)

1. **Why do we filter to productive sequences?**
   - Only productive sequences actually function
   - Non-productive sequences are errors

2. **What are valid amino acids?**
   - 20 standard amino acids: A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y
   - Any other character is invalid

3. **Why aggregate duplicates?**
   - Same amino acid sequence = same TCR function
   - We want to count T cells, not DNA sequences

4. **What does `groupby().agg()` do?**
   - Groups rows by certain columns
   - Applies a function (like sum) to other columns
   - Like Excel pivot tables!

## Step 1: Load Data

Let's start by loading the data again.

**Biology context:**
- This is raw TCR-seq data from one patient
- Contains all sequences, including errors
- We'll clean it step by step

**CS context:**
- Load TSV file with pandas
- We'll track how many sequences we have at each step
- This helps us see what each cleaning step does

**Your Task:** Load the data file

In [ ]:
# Import pandas
import pandas as pd
from pathlib import Path
import os

# Find data directory (works in both local + Colab once files are uploaded)
data_dir = Path("../data/DeepTCR_Cancer-master/Data/yost/data")
if not data_dir.exists():
    alt_paths = [
        Path("data/DeepTCR_Cancer-master/Data/yost/data"),
        Path("./data/DeepTCR_Cancer-master/Data/yost/data"),
    ]
    for alt_path in alt_paths:
        if alt_path.exists():
            data_dir = alt_path
            print(f"✓ Found data at: {alt_path}")
            break

file_path = data_dir / "su001_BCC_pre1_TCRB.tsv"

# ✅ Starter solution
# If you haven't written the TODO yet, we'll load a small sample for you.
auto_loaded = False
if ("df" not in globals()) or (df is None):
    if file_path.exists():
        df = pd.read_csv(file_path, sep="\t")
        auto_loaded = True
    else:
        df = None
        print("✗ Could not find the example file. Double-check that the data folder is available.")

if df is not None:
    starting_count = len(df)
    print(f"✓ Loaded {starting_count:,} sequences from {file_path.name}")
    print("  This is the RAW data - includes errors and duplicates")
    if auto_loaded:
        print("  Tip: Replace the starter code above with your own call to pd.read_csv() once you're ready.")
else:
    print("⚠️ dataframe `df` is still None. Use pd.read_csv(file_path, sep='\\t') to load it.")

## Step 2: Filter to Productive Sequences

We only want sequences that are "In" frame (productive).

**Biology reason:**
- Only "In" frame sequences actually function
- "Out" frame sequences are broken
- We want functional T cells only!

**CS reason:**
- Filtering removes irrelevant data
- Makes analysis cleaner
- Uses pandas boolean indexing: `df[df['col'] == 'value']`

**Your Task:** Filter to only productive sequences

In [ ]:
# Filter to only 'In' frame sequences
if df is None:
    raise ValueError("Run the previous cell to load `df` before filtering.")

# ✅ Starter solution (runs unless you've already created `productive`)
if ("productive" not in globals()) or (productive is None):
    productive = df[df["sequenceStatus"] == "In"].copy()
    print("✓ Using built-in helper to keep only productive sequences.")
else:
    print("✓ Using your custom `productive` DataFrame.")

productive_count = len(productive)
kept_pct = (productive_count / starting_count) * 100 if starting_count else 0

print("\nFiltering results:")
print(f"  Starting: {starting_count:,} sequences")
print(f"  Productive: {productive_count:,} sequences")
print(f"  Removed: {starting_count - productive_count:,} sequences")
print(f"  Kept: {kept_pct:.1f}%")

## Step 3: Remove Invalid Amino Acids

Valid amino acids are: A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y (20 total)

**Biology reason:**
- There are only 20 standard amino acids
- Any other character (X, *, -, etc.) is an error
- Invalid sequences are sequencing errors, not real TCRs

**CS reason:**
- We need to validate each sequence
- Use `.apply()` to check each row
- Filter to only valid sequences

**How it works:**
- `.apply(lambda x: ...)` runs a function on each row
- `all(c in valid_aa for c in str(x))` checks if all characters are valid
- Returns True if valid, False if invalid

**Your Task:** Remove sequences with invalid characters

In [ ]:
# Define valid amino acids
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
print(f"✓ Valid amino acids: {len(valid_aa)}")
print(f"  {''.join(sorted(valid_aa))}")
print("\nAny other character (X, *, -, etc.) is INVALID")

if "productive" not in globals():
    raise ValueError("Run the previous cell to create the `productive` DataFrame first.")

# ✅ Starter validation
if ("is_valid" not in globals()) or (is_valid is None):
    is_valid = productive["aminoAcid"].apply(lambda seq: isinstance(seq, str) and all(aa in valid_aa for aa in seq))
    print("✓ Auto-generated boolean mask `is_valid`.")
else:
    print("✓ Using your custom `is_valid` mask.")

if ("valid" not in globals()) or (valid is None):
    valid = productive[is_valid].copy()
    print("✓ Filtered to valid amino acid sequences using helper code.")
else:
    print("✓ Using your custom filtered DataFrame `valid`.")

valid_count = len(valid)
print("\nValidation results:")
print(f"  Before: {productive_count:,} sequences")
print(f"  After: {valid_count:,} sequences")
print(f"  Removed: {productive_count - valid_count:,} invalid sequences")
print(f"  Kept: {valid_count / productive_count * 100:.1f}%")

## Step 4: Filter by Sequence Length

TCR sequences are usually 10-25 amino acids long. Let's remove unusual lengths.

**Biology reason:**
- Functional TCR sequences are typically 10-25 amino acids
- Very short sequences (< 10) are incomplete or errors
- Very long sequences (> 25) are unusual and might be errors
- **Why it matters:** Unusual lengths are likely artifacts

**CS reason:**
- Calculate length for each sequence
- Filter using boolean conditions
- Use `&` (and) to combine conditions

**How it works:**
- `df['col'].str.len()` gets length of each string
- `(lengths >= 10) & (lengths <= 25)` creates boolean array
- `df[boolean_array]` filters to True rows

**Your Task:** Filter to sequences with length 10-25

In [ ]:
if "valid" not in globals():
    raise ValueError("Make sure the previous cell defined the `valid` DataFrame before continuing.")

# ✅ Starter length filtering
if ("lengths" not in globals()) or (lengths is None):
    lengths = valid["aminoAcid"].str.len()
    print("✓ Calculated sequence lengths with starter code.")
else:
    print("✓ Using your custom `lengths` Series.")

print("Before filtering:")
print(f"  Min length:  {lengths.min()}")
print(f"  Max length:  {lengths.max()}")
print(f"  Mean length: {lengths.mean():.2f}")

if ("df_clean" not in globals()) or (df_clean is None):
    df_clean = valid[(lengths >= 10) & (lengths <= 25)].copy()
    print("\n✓ Applied helper filter to keep lengths between 10 and 25 amino acids.")
else:
    print("\n✓ Using your custom cleaned DataFrame `df_clean`.")

filtered_count = len(df_clean)
print("\nAfter filtering:")
print(f"  Min length:  {df_clean['aminoAcid'].str.len().min()}")
print(f"  Max length:  {df_clean['aminoAcid'].str.len().max()}")
print(f"  Mean length: {df_clean['aminoAcid'].str.len().mean():.2f}")

print("\nLength filtering results:")
print(f"  Before: {valid_count:,} sequences")
print(f"  After: {filtered_count:,} sequences")
print(f"  Removed: {valid_count - filtered_count:,} sequences")
print(f"  Kept: {filtered_count / valid_count * 100:.1f}%")

## Step 5: Aggregate Duplicate Sequences

The same sequence might appear multiple times (different DNA sequences can encode the same amino acid sequence).

**Biology reason:**
- Multiple DNA sequences → same amino acid sequence = same TCR function
- We want to count how many T cells have this TCR, not how many DNA sequences
- **Why it matters:** We care about TCR function (amino acids), not DNA sequence

**CS reason:**
- Use `groupby()` to group rows with same values
- Use `.agg()` to sum the counts
- Like Excel pivot tables!

**How it works:**
- `df.groupby(['col1', 'col2', 'col3'])` groups rows by these columns
- `.agg({'count_col': 'sum'})` sums the count column for each group
- `.reset_index()` converts back to regular DataFrame

**Your Task:** Group by sequence + V gene + J gene, and sum the counts
**💡 Hint (Click to expand):**

<details>
<summary>Show example groupby pattern</summary>

```python
# Example pattern:
df_aggregated = (
    df_clean
    .groupby(['aminoAcid', 'vGeneName', 'jGeneName'], dropna=False)
    .agg({'count (templates/reads)': 'sum'})  # Sum counts for duplicates
    .reset_index()  # Convert index back to columns
)
```

**What this does:**
- Groups rows with same (aminoAcid, vGeneName, jGeneName)
- Sums the counts for each group
- Resets index to get normal DataFrame

</details>



In [ ]:
if "df_clean" not in globals():
    raise ValueError("Run the length-filtering cell to create `df_clean` first.")

before_agg = len(df_clean)
print(f"Before aggregation: {before_agg:,} sequences")
print("  (Some sequences appear multiple times)")

# ✅ Starter aggregation
if ("df_aggregated" not in globals()) or (df_aggregated is None):
    df_aggregated = (
        df_clean
        .groupby(["aminoAcid", "vGeneName", "jGeneName"], dropna=False)
        .agg({"count (templates/reads)": "sum"})
        .reset_index()
    )
    print("✓ Combined duplicate sequences (same amino acid + V gene + J gene).")
else:
    print("✓ Using your custom grouped DataFrame `df_aggregated`.")

after_agg = len(df_aggregated)
print("\nAggregation results:")
print(f"  Before: {before_agg:,} sequences")
print(f"  After: {after_agg:,} unique sequences")
print(f"  Removed: {before_agg - after_agg:,} duplicate entries")
print(f"  Kept: {after_agg / before_agg * 100:.1f}% unique sequences")

## Step 6: Summary of Cleaning Steps

Let's see the final cleaned data and summarize what we did.

**Biology summary:**
- Removed non-functional sequences (non-productive)
- Removed sequencing errors (invalid amino acids)
- Removed artifacts (unusual lengths)
- Aggregated duplicates (count T cells, not DNA sequences)

**CS summary:**
- Filtered data multiple times
- Validated sequences
- Aggregated duplicates
- Result: Clean DataFrame ready for analysis!

**Your Task:** Print summary and show cleaned data

In [ ]:
required_vars = ["starting_count", "productive_count", "valid_count", "filtered_count", "after_agg", "df_aggregated"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise ValueError(f"Run the earlier cells first. Missing variables: {missing}")

print("\n" + "=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"1. Starting: {starting_count:,} sequences")
print(f"2. After filtering productive: {productive_count:,} sequences")
print(f"3. After removing invalid: {valid_count:,} sequences")
print(f"4. After length filtering: {filtered_count:,} sequences")
print(f"5. After aggregation: {after_agg:,} unique sequences")
print(f"\nFinal: {after_agg:,} clean sequences")
removed_total = starting_count - after_agg
print(f"Removed: {removed_total:,} sequences ({(removed_total / starting_count) * 100:.1f}%)")

print("\nPeek at the cleaned data:")
print(df_aggregated.head())
print(f"\nShape of cleaned data: {df_aggregated.shape}")
print("\nBasic stats for counts (templates/reads):")
print(df_aggregated["count (templates/reads)"].describe())

## ✅ Summary: What You Learned Today

### Biology:
- ✅ Why we filter to productive sequences (only functional TCRs)
- ✅ Why we remove invalid amino acids (sequencing errors)
- ✅ Why we filter by length (remove artifacts)
- ✅ Why we aggregate duplicates (count T cells, not DNA)

### Computer Science:
- ✅ How to filter DataFrames (`df[condition]`)
- ✅ How to validate data (`.apply()` with lambda)
- ✅ How to aggregate data (`groupby().agg()`)
- ✅ How to track cleaning steps

### Pandas Operations:

1. `df[df['col'] == 'value']` - Filtering
2. `df['col'].apply(function)` - Apply function to each row
3. `df['col'].str.len()` - Get string lengths
4. `df[(condition1) & (condition2)]` - Multiple conditions
5. `df.groupby(['col1', 'col2']).agg({'col3': 'sum'})` - Group and aggregate
6. `.reset_index()` - Convert index back to columns

**All of these are pandas operations you already know!**

## 🎉 You're Done with Day 3!

**What you accomplished:**
- ✅ Filtered to productive sequences
- ✅ Removed invalid amino acids
- ✅ Filtered by length
- ✅ Aggregated duplicates
- ✅ Created clean data ready for analysis!
- ✅ Learned biology AND CS context!

**Next Step:** Go to `Day_04_Understand_Data/Day_04_Understand_Data.ipynb`

## 📝 Notes Section

Write down:
- What each cleaning step does (in your own words)
- Why each step is important (biology + CS)
- How many sequences you removed at each step
- Questions you still have